# 03 — Customer Churn Analysis

Deep dive into the XGBoost churn prediction model — confusion matrix, ROC curve, and feature importance.

## Setup

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, sys
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    classification_report, ConfusionMatrixDisplay
)
from sqlalchemy import create_engine, text
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from config import PG_URL, MODELS_DIR
engine = create_engine(PG_URL)
plt.rcParams['figure.figsize'] = (12, 5)
print("✅ Setup complete")


✅ Setup complete


## 1. Load Features & Model

In [2]:

sql = '''
SELECT f.customer_id,
    DATE '2018-09-01' - MAX(f.order_date)::date   AS recency_days,
    COUNT(DISTINCT f.order_id)                    AS frequency,
    ROUND(SUM(f.revenue)::numeric,2)              AS monetary,
    ROUND(AVG(f.review_score)::numeric,2)         AS avg_review_score,
    ROUND(AVG(f.freight_value/NULLIF(f.revenue,0))::numeric,4) AS avg_freight_pct,
    ROUND(AVG(f.payment_installments)::numeric,2) AS avg_installments,
    SUM(CASE WHEN f.delivery_delay_days>0 THEN 1 ELSE 0 END) AS late_deliveries,
    ROUND(AVG(f.delivery_delay_days)::numeric,1)  AS avg_delay_days,
    COUNT(DISTINCT f.category)                    AS unique_categories,
    COUNT(DISTINCT f.product_id)                  AS unique_products
FROM olist.fact_orders f WHERE f.is_delivered=1
GROUP BY f.customer_id
'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql), conn)

df['churned'] = (df['recency_days'] > 180).astype(int)
features = ['recency_days','frequency','monetary','avg_review_score',
            'avg_freight_pct','avg_installments','late_deliveries',
            'avg_delay_days','unique_categories','unique_products']
X = df[features].fillna(0)
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = joblib.load(os.path.join(MODELS_DIR, 'churn_scaler.pkl'))
model  = joblib.load(os.path.join(MODELS_DIR, 'churn_model.pkl'))
X_test_sc = scaler.transform(X_test)
y_pred    = model.predict(X_test_sc)
y_proba   = model.predict_proba(X_test_sc)[:,1]
print("✅ Model and data loaded")
print(classification_report(y_test, y_pred, target_names=['Active','Churned']))


FileNotFoundError: [Errno 2] No such file or directory: 'd:\\Programming\\Python\\Data Science\\Olist Brazilian E-Commerce\\models\\churn_scaler.pkl'

## 2. Confusion Matrix

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Active','Churned'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix')
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Churn Model')
axes[1].legend()
plt.tight_layout()
plt.show()


## 3. Feature Importance

In [ ]:

importance = pd.DataFrame({
    'feature'   : features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

plt.barh(importance['feature'], importance['importance'], color='steelblue')
plt.title('XGBoost Feature Importance — Churn Model')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()
print("
Top 3 most important features:")
for _, row in importance.tail(3)[::-1].iterrows():
    print(f"  {row['feature']:<25} {row['importance']:.4f}")


## 4. High Risk Customers

In [ ]:

df_test = X_test.copy()
df_test['churn_probability'] = y_proba
df_test['actual_churned']    = y_test.values
df_test['risk_level'] = df_test['churn_probability'].apply(
    lambda p: 'High' if p>=0.7 else 'Medium' if p>=0.4 else 'Low'
)

print("Risk Level Distribution:")
print(df_test['risk_level'].value_counts().to_string())
print(f"
Top 10 highest churn risk customers:")
top10 = df_test.nlargest(10, 'churn_probability')[
    ['recency_days','frequency','monetary','churn_probability','risk_level']
]
print(top10.to_string(index=False))
